In [ ]:
import matplotlib.pyplot as plt
import math

def create_adjacency_list(edges):
    graph = {}
    for u, v in edges:
        if u not in graph: graph[u] = set()
        if v not in graph: graph[v] = set()
        graph[u].add(v)
        graph[v].add(u)
    return graph

def lex_bfs(graph):
    labels = {v: [] for v in graph}
    unnumbered = set(graph.keys())
    sigma = []
    n = len(graph)
#serce największa etykieta leksykograficznie
    for i in range(n, 0, -1):
        v = max(unnumbered, key=lambda x: labels[x])
        sigma.append(v)
        unnumbered.remove(v)
#jeśli nie został przetworzony zwiększamy priorytet
        for u in graph[v]:
            if u in unnumbered:
                labels[u].append(i)
    return sigma


def is_chordal(graph, peo):

    pos = {v: i for i, v in enumerate(peo)}

    for v in peo:
        later_neighbors = [n for n in graph[v] if pos[n] > pos[v]]

        if not later_neighbors:
            continue

        later_neighbors.sort(key=lambda x: pos[x])
        u = later_neighbors[0]

        for w in later_neighbors[1:]:
            if w not in graph[u]:
                return False

    return True


def greedy_coloring(graph, order):
    coloring = {}
    for vertex in order:
        used_colors = set()
        for neighbor in graph[vertex]:
            if neighbor in coloring:
                used_colors.add(coloring[neighbor])

        color = 0
        while color in used_colors:
            color += 1
        coloring[vertex] = color

    return coloring


def draw_graph_matplotlib(graph, coloring):
    nodes = list(graph.keys())
    n = len(nodes)

    positions = {}
    for i, node in enumerate(nodes):
        angle = 2 * math.pi * i / n
        positions[node] = (math.cos(angle), math.sin(angle))

    fig, ax = plt.subplots(figsize=(6, 6))

    for u in graph:
        for v in graph[u]:
            if u < v:
                x_values = [positions[u][0], positions[v][0]]
                y_values = [positions[u][1], positions[v][1]]
                ax.plot(x_values, y_values, color='gray', zorder=1, linewidth=1.5)

    cmap = plt.get_cmap('Set2')
    for node in nodes:
        x, y = positions[node]
        c = cmap(coloring.get(node, 0) % 8)

        ax.scatter(x, y, s=1200, color=c, zorder=2, edgecolors='black')
        ax.text(x, y, str(node), ha='center', va='center',
                color='black', fontweight='bold', fontsize=12, zorder=3)

    ax.axis('off')
    plt.title("Kolorowanie grafu - LexBFS")
    plt.show()




edges = [
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 1),
    (4, 2)
]

graph = create_adjacency_list(edges)

lex_bfs_order = lex_bfs(graph)

peo = lex_bfs_order[::-1]

print("Kolejność LexBFS (do kolorowania):", lex_bfs_order)
print("PEO (do sprawdzenia cięciwowości):", peo)

if is_chordal(graph, peo):
    print("\nGraf JEST cięciwowy.")


    coloring = greedy_coloring(graph, lex_bfs_order)

    print("\nWynik kolorowania (wierzchołek -> kolor):")
    for v in sorted(coloring.keys()):
        print(f"{v} -> {coloring[v]}")

    draw_graph_matplotlib(graph, coloring)

else:
    print("\nGraf NIE JEST cięciwowy.")